# Introduction to NumPy & Vectorization (Prep for PyTorch)

Welcome! Deep Learning relies entirely on manipulating large multidimensional arrays (tensors). Before stepping into PyTorch, we need to master **NumPy**, the foundation of scientific computing in Python.

By the end of this notebook, you will understand:
1. Why we avoid Python `for` loops (Vectorization).
2. How to map operations across dimensions (Broadcasting).
3. Extracting and filtering data efficiently (Indexing & Masking).
4. Collapsing dimensions (Reductions & Axes).
5. Morphing array structures (Reshaping).

In [ ]:
import numpy as np
import time
import math

## 1. Why Vectorization? (Python Lists vs. NumPy)
In Python, standard `for` loops are slow when processing massive amounts of data. NumPy pushes these operations down to optimized C code. Let's see the difference.

In [ ]:
# EXAMPLE: Adding two large sets of numbers
size = 1_000_000
list_a = list(range(size))
list_b = list(range(size))

arr_a = np.arange(size)
arr_b = np.arange(size)

# Python for-loop
start = time.time()
list_c = [a + b for a, b in zip(list_a, list_b)]
print(f"Python List comprehension took: {time.time() - start:.4f} seconds")

# NumPy Vectorization
start = time.time()
arr_c = arr_a + arr_b
print(f"NumPy Vectorization took:       {time.time() - start:.4f} seconds")

### Exercise 1: Element-wise Operations
**Task**: You have two large arrays representing predictions and targets. Calculate the Mean Squared Error (MSE) between them.
$MSE = \frac{1}{n} \sum (predictions - targets)^2$

*Hint: Do this completely without loops!*

In [ ]:
np.random.seed(42)
preds = np.random.rand(500_000)
targets = np.random.rand(500_000)

# TODO: Calculate MSE between preds and targets without loops
mse = ...
print(f"MSE: {mse}")

## 2. Broadcasting (The Magic of NumPy)
Broadcasting allows NumPy to perform operations on arrays of different shapes. The smaller array is "broadcast" across the larger array so that they have compatible shapes.

**Rule of thumb**: Dimensions are compatible when:
1. They are equal, OR
2. One of them is 1.

In [ ]:
# EXAMPLE: Center a dataset around 0
# Pretend this is a batch of 5 items, each with 3 features (e.g., RGB values)
batch = np.array([
    [10, 20, 30],
    [12, 22, 32],
    [8, 18, 28],
    [11, 21, 31],
    [9, 19, 29]
]) # Shape: (5, 3)

feature_means = np.array([10, 20, 30]) # Shape: (3,) -> treated as (1, 3)

# Broadcasting automatically expands feature_means to (5, 3) to subtract!
centered_batch = batch - feature_means
print("Centered Batch:\n", centered_batch)

print(f"Shape of Batch: {batch.shape}, Shape of Feature Means: {feature_means.shape}")
print("Shape of Centered Batch:", centered_batch.shape)

### The `None` Dimension (`np.newaxis`)
In the next example, notice the use of `None` inside the brackets. In NumPy (and identically in PyTorch), using `None` during indexing is a very common shortcut for creating a brand new dimension of size 1 (this is exactly equivalent to using `np.newaxis` or calling `reshape`/`unsqueeze`).

For example, if `batch` has shape `(5, 3)`, slicing it with `batch[:, None, :]` transforms its shape to `(5, 1, 3)`. This trick is incredibly useful for manually aligning array dimensions in a single line so broadcasting can match them up!

In [ ]:
feature_means_2 = np.array([
    [10, 20, 30],
    [10, 20, 30],
]) # Shape: (2, 3)

batch[:, None, :] - feature_means_2[None, :, :] # Shape: (5, 1, 3) - (1, 2, 3) -> (5, 2, 3)
batch[None, :, :] - feature_means_2[:, None, :] # Shape: (1, 5, 3) - (2, 1, 3) -> (2, 5, 3)

print(f"Shape of batch[:, None, :]: {batch[:, None, :].shape}, Shape of feature_means_2[None, :, :]: {feature_means_2[None, :, :].shape}")

### Exercise 2: Image Normalization
**Task**: Deep learning practically always requires normalizing your images. You are given a batch of 100 images. Each image is 32x32 pixels with 3 color channels (Shape: `100, 32, 32, 3`). Let's normalize it by multiplying every pixel in the Red channel by 0.5, the Green channel by 0.2, and the Blue channel by 0.8.

*Hint: Construct an array of shape `(3,)` containing your scales, and multiply it directly with the image batch!*

In [ ]:
image_batch = np.ones((100, 32, 32, 3)) # Dummy data

# TODO: Multiply channels by R:0.5, G:0.2, B:0.8 using broadcasting
channel_scales = ...
scaled_images = ...

print("New shape:", scaled_images.shape)
print("Pixel 0,0 of Image 0:", scaled_images[0, 0, 0, :]) # Should be [0.5, 0.2, 0.8]

## 3. Reductions and Axes (Collapsing Dimensions)
Often, you need to collapse a multidimensional array (e.g., sum all columns, get max of each row). You do this via the `axis` parameter.
*   `axis=0`: Operations go DOWN the rows (collapses rows).
*   `axis=1`: Operations go ACROSS the columns (collapses columns).

In [ ]:
# EXAMPLE
scores = np.array([
    [90, 80, 85],   # Student 1 scores (Math, Science, English)
    [60, 70, 75],   # Student 2 scores
    [100, 95, 92]   # Student 3 scores
])

print("Average per student (axis=1):", np.mean(scores, axis=1))
print("Average per subject (axis=0):", np.mean(scores, axis=0))

### Exercise 3: Softmax Denominator (Crucial for ML)
**Task**: In Deep Learning classification, the Softmax function requires summing the exponentials of predictions *per sample* in a batch.
Given a batch of 4 samples and 3 classes `(4, 3)`, compute the sum of `np.exp()` for each sample (you should get 4 numbers).

*Important trick*: Pass the `keepdims=True` parameter to your sum function. Look at the resulting shape. What happens if you don't use it?

In [ ]:
logits = np.array([[2.0, 1.0, 0.1],
                   [0.5, 2.5, 0.2],
                   [0.1, 0.1, 0.1],
                   [3.0, 0.0, -1.0]])

# TODO: Compute sum of exponentials along the classes (axis=1) with keepdims=True
exp_sum = ...

print("Exp sum shape:", exp_sum.shape) # Should be (4, 1)

## 4. Indexing, Slicing and Masking
NumPy offers powerful ways to extract subsets of arrays.
- **Slicing**: Extracting a range of elements (e.g. `array[start:stop:step]`)
- **Masking**: Filtering values based on conditions. Think of it as passing a True/False array as an index.

In [ ]:
# EXAMPLE: Slicing
matrix = np.array([
    [1, 2, 3, 4],
    [5, 6, 7, 8],
    [9, 10, 11, 12]
])

print("Original Matrix:\n", matrix)
print("\nGet the first 2 rows, and the last 2 columns:\n", matrix[:2, 2:])
print("\nGet every other element of the second row:\n", matrix[1, ::2])

In [ ]:
# EXAMPLE: Find all even numbers
arr = np.array([1, 2, 3, 4, 5, 6])
mask = (arr % 2 == 0)
print("Mask:", mask, "mask type:", type(mask), "mask shape:", mask.shape, "mask dtype:", mask.dtype)
print("Even numbers:", arr[mask])

### Exercise 4: Building a ReLU Activation Function
**Task**: The most popular activation function in neural networks is ReLU (Rectified Linear Unit). It simply takes any negative number and sets it to 0, leaving positive numbers unchanged.

Given the array `x`, modify it in-place using boolean masking so that all negative values become 0.

In [ ]:
x = np.array([-2.5, 3.1, -0.1, 5.0, -8.3, 1.1])

# TODO: Use masking to set values < 0 to 0
...

print("ReLU'd x:", x)

## 5. Reshaping & Combining Arrays (Morphing)
Neural networks are notoriously picky about input shapes. `reshape()` and `.T` (Transpose) are your best friends.
When reshaping, one dimension can be `-1`. NumPy will automatically figure out how big that dimension needs to be!

Additionally, you frequently need to stitch arrays together using `concatenate()` (joins existing axes) or `stack()` (creates a new axis).

In [ ]:
# EXAMPLE:
a = np.arange(12) # shape (12,)
print(a.reshape(3, 4)) # 3 rows, 4 cols
print("\nUsing -1:")
print(a.reshape(2, -1)) # 2 rows, NumPy calculates it needs 6 cols

In [ ]:
# EXAMPLE: Concatenation and Stacking
a = np.array([1, 2])
b = np.array([3, 4])

print("Concatenate (joins along existing dimension):")
print(np.concatenate([a, b]))  # Shape: (4,)

print("\nStack (creates a brand NEW dimension):")
print(np.stack([a, b]))        # Shape: (2, 2)

### Exercise 5: Flattening an Image Dataset
**Task**: Fully connected neural networks (MLPs) can't take 2D images. They need a 1D vector.
You are given a batch of 50 grayscale images, each 28x28 (Shape: `50, 28, 28`).
Reshape this array so that the batch size (50) is kept as the first dimension, and the spatial dimensions (28x28) are flattened into a single dimension of 784.

In [ ]:
dataset = np.ones((50, 28, 28))

# TODO: Reshape dataset to (50, 784). Try using -1 in your reshape call!
flattened = ...

print("Flattened shape:", flattened.shape) # Should be (50, 784)

---
### You're Ready for PyTorch!
The grand secret of Deep Learning is that **PyTorch is effectively NumPy but runs on a GPU.**
The APIs are deliberately nearly identical.
`np.sum()` -> `torch.sum()`
`np.zeros()` -> `torch.zeros()`
`np.reshape()` -> `torch.reshape()`

Great job reaching the end of the primer!

# Part II: Introduction to PyTorch

Welcome to PyTorch! You'll notice immediately that **PyTorch Tensors are heavily inspired by NumPy arrays**, and many of the operations are identical. The two huge differences are:
1. PyTorch tensors can be pushed to GPUs for massive training speedups.
2. PyTorch tracks operations on tensors to automatically calculate gradients (Autograd) for neural networks (which we will cover next!).

Let's look at the basics.

In [ ]:
import torch

## 6. Creating Tensors & The NumPy Bridge
You can create tensors from scratch, or convert to and from NumPy directly. Under the hood, a CPU PyTorch tensor and a NumPy array can even share the same physical memory, so conversions are completely free!

In [ ]:
# EXAMPLE: Creating Tensors
print("Zeros:\n", torch.zeros(2, 3))
print("Random:\n", torch.rand(2, 3))

# The NumPy Bridge
np_arr = np.array([[1, 2], [3, 4]])
pt_tensor = torch.from_numpy(np_arr)

print("\nPyTorch Tensor from NumPy:\n", pt_tensor)
print("Tensor Data Type:", pt_tensor.dtype)
print("Tensor Shape:", pt_tensor.shape)

### More Ways to Initialize Tensors
Just like NumPy, PyTorch has many built-in initialization methods. Here are a few common ones:

In [ ]:
# Uninitialized matrix (allocates memory without overriding previous values)
empty_tensor = torch.empty(2, 2)

# Identity matrix (1s on the diagonal)
eye_tensor = torch.eye(3)

# Create a tensor mirroring the shape and type of an existing tensor
ones_like = torch.ones_like(eye_tensor)

print("Empty:\n", empty_tensor)
print("\nEye:\n", eye_tensor)
print("\nOnes Like:\n", ones_like)

### Basic Torch Attributes
Every tensor has 3 key attributes you will frequently check:
- `.shape` (or `.size()`): The dimensions of the tensor
- `.dtype`: The mathematical type of the data (float32, int64, etc.)
- `.device`: Where the data lives (CPU or GPU)

In [ ]:
# EXAMPLE: Inspecting Attributes
sample_tensor = torch.zeros(3, 2)

print("Shape:", sample_tensor.shape)  # Equivalent to sample_tensor.size()
print("Data Type:", sample_tensor.dtype)
print("Device:", sample_tensor.device)

PyTorch supports several different data types for tensors, each of which can be specified using the `dtype` argument when creating a tensor. Here are some of the most commonly used data types:

* `torch.float32`: 32-bit floating-point number (float)
* `torch.float64`: 64-bit floating-point number (double)
* `torch.float16`: 16-bit floating-point number (half-precision)
* `torch.int8`: 8-bit integer (signed)
* `torch.uint8`: 8-bit integer (unsigned)
* `torch.int16`: 16-bit integer (signed)
* `torch.int32`: 32-bit integer (signed)
* `torch.int64`: 64-bit integer (signed)
* `torch.bool`: boolean (True or False)

### Exercise 6: Tensor Types
**Task**: Create a PyTorch tensor filled with ones of shape `(3, 3)`. By default, it will be `torch.float32`. Create a second tensor also of ones, but change its data type to `torch.int32`.

*Hint:* Use `torch.ones()` and its `dtype` argument.

In [ ]:
# TODO: Create a 3x3 float32 tensor of ones
float_ones = ...

# TODO: Create a 3x3 int32 tensor of ones
int_ones = ...

print("Float dtype:", float_ones.dtype)
print("Int dtype:", int_ones.dtype)

## 7. Tensor Math, Transpose, Inverse & .item()
The operations you learned in NumPy (`+`, `*`, `@`, methods like `.mean()`, `.sum()`, etc.) function natively on PyTorch tensors.
For reshaping arrays, PyTorch typically uses `.view()` or `.reshape()`. They are essentially the same, but `.view()` is strict about ensuring the underlying memory is contiguous.

In [ ]:
# EXAMPLE: Matrix Multiplication, Transpose, and Inverse
tensor_a = torch.randn(3, 3)
tensor_b = torch.randn(3, 3)

# Multiplication
print("Element-wise multiply:", (tensor_a * tensor_b).shape)
print("Matrix multiply (@):", (tensor_a @ tensor_b).shape) # Or tensor_a.matmul(tensor_b)

# Transpose and Inverse
print("\nTranspose:", tensor_a.t().shape) # Or tensor_a.T
print("Inverse:", torch.inverse(tensor_a).shape)

# .item() - Used heavily to get the raw Python float out of a 1-element tensor (like loss!)
loss_tensor = torch.tensor([4.25])
print("\nRaw python float from .item():", loss_tensor.item())

# Reshaping with .view()
flattened = tensor_a.view(-1)
print("\nFlattened using view(-1):", flattened.shape)

### Memory Contiguity: `.view()` vs `.reshape()`
Under the hood, PyTorch tries to save memory by having multiple tensors share the same raw underlying data.
- `.view()` returns a reshaped tensor sharing the same data. It is very fast, but it **requires** the tensor's memory to be "contiguous" (laid out sequentially without gaps).
- Operations like transpose (`.T` or `.t()`) change how PyTorch determines the dimensions, but they *do not change the raw data block in memory*. This makes the resulting tensor **non-contiguous**.
- `.reshape()` is the robust alternative. If a tensor is contiguous, it acts exactly like `.view()`. If it is non-contiguous, it will automatically allocate a new continuous memory block (making a physical copy of the data) and then reshape it!

In [ ]:
# Create a standard contiguous tensor
t = torch.arange(6).view(2, 3)
print("Is t contiguous?", t.is_contiguous())

# Transposing changes metadata, but not the raw memory -> Non-contiguous!
t_t = t.t()
print("Is t_t contiguous?", t_t.is_contiguous())

try:
    # This will crash!
    t_t.view(-1)
except RuntimeError as e:
    print("\nCaught Error with .view():", e)

# Reshape handles it perfectly (by making a contiguous copy under the hood)
safe_reshape = t_t.reshape(-1)
print("\nSuccessful .reshape():", safe_reshape)

# If you strictly want to use .view(), you must explicitly make it contiguous first
safe_view = t_t.contiguous().view(-1)
print("Successful .contiguous().view():", safe_view)

### In-Place vs. Standard Operations
Most PyTorch operations have an "in-place" variant denoted by an underscore `_` at the end of the method name. This alters the tensor directly in memory rather than creating a new copy, which can be great for saving RAM!

In [ ]:
x = torch.ones(2, 2)
y = torch.ones(2, 2) * 2

# Standard addition (creates a brand new tensor)
z = x + y
# Alternatively: z = torch.add(x, y)

# In-place addition: adds 'x' directly to 'y' and modifies 'y' in memory
# TODO : Talk about gradients and in-place operations in the context of autograd
y.add_(x)

print("z (New Tensor):\n", z)
print("\ny (Modified In-Place Tensor):\n", y)

In [ ]:
# EXAMPLE: Slicing (Same as NumPy!)
tensor = torch.tensor([
    [10, 20, 30],
    [40, 50, 60],
    [70, 80, 90]
])

print("Original Tensor:\n", tensor)
print("\nFirst column:", tensor[:, 0])
print("Last two rows:\n", tensor[-2:])

### Exercise 7: Masking in PyTorch
**Task**: Masking works the exact same way it does in NumPy! Given the random tensor `x`, use a boolean mask to replace all values greater than `0.5` with `1.0`, and keep the rest unchanged.

In [ ]:
torch.manual_seed(42)  # For reproducibility
x = torch.rand(5)
print("Original x:", x)

# TODO: Replace values in x > 0.5 with 1.0
...

print("Modified x:", x)

## 8. Dimensionality: Squeeze, Unsqueeze & Combining
Sometimes tensors have annoying dimensions of size 1 (e.g. shape `(10, 1, 5)`), or you need to specifically inject a dimension of size 1 to make broadcasting work.
- `unsqueeze(dim)`: Inserts a dimension of size 1.
- `squeeze(dim)`: Removes a dimension of size 1.

PyTorch also has `torch.cat` and `torch.stack`, behaving exactly like their NumPy equivalents from earlier.

In [ ]:
# EXAMPLE: Squeeze & Unsqueeze
x = torch.zeros(3, 5)

print("Original shape:", x.shape)
print("Unsqueeze at dim 0:", x.unsqueeze(0).shape) # (1, 3, 5)
print("Squeeze back:", x.unsqueeze(0).squeeze(0).shape)  # Back to (3, 5)

# EXAMPLE: Cat and Stack
t1 = torch.ones(2, 2)
t2 = torch.zeros(2, 2)

print("\nCat (dim=0):", torch.cat([t1, t2], dim=0).shape) # (4, 2)
print("Stack (dim=0):", torch.stack([t1, t2], dim=0).shape) # (2, 2, 2)

### Exercise 8: From Individuals to a Batch
**Task**: In reinforcement learning or image processing, you often process items individually in a loop, yielding a standard Python list of tensors. Before passing them through a network, you must convert that list into a single batched tensor!

Given a list containing four tensors of shape `(3, 3)`, combine them so that the final tensor has shape `(4, 3, 3)`.  
*Hint: Should you use `torch.cat` or `torch.stack`?*

In [ ]:
images = [torch.rand(3, 3) for _ in range(4)]
print(f"List contains {len(images)} tensors of shape {images[0].shape}\n")

# TODO: Combine the list of tensors into a single batched tensor of shape (4, 3, 3)
batched = ...

print("Batched Shape:", batched.shape)

## 8. PyTorch's Superpower: The GPU
By default, tensors are created on the CPU. To accelerate math operations, we explicitly move them to a GPU (if one is available).

If you are running this in Google Colab, you can hook up a free GPU via `Runtime > Change runtime type > Hardware accelerator > T4 GPU`.

In [ ]:
# EXAMPLE: Checking for GPU
if torch.cuda.is_available():
    print("Awesome! A GPU is available.")
    device = torch.device("cuda") # Specify Nvidia GPU
else:
    print("No GPU available, running on CPU. (If in Colab, check your runtime settings!)")
    device = torch.device("cpu")

### CPU vs GPU Performance Benchmark
Let's practically see why Deep Learning relies on GPUs. We will multiply two massive `5000 x 5000` matrices together.
Matrix multiplication is highly parallelizable, which makes it the absolute perfect job for a GPU containing thousands of tiny cores.

*Note: We must use `torch.cuda.synchronize()` when timing GPU operations because PyTorch dispatches GPU operations asynchronously! If we didn't synchronize, the timer would stop before the GPU even finished the calculation!*

In [ ]:
import time

# Create two huge matrices (5000x5000 = 25 million elements each!)
size = 5000
a_cpu = torch.randn(size, size)
b_cpu = torch.randn(size, size)

print("Timing CPU Matrix Multiplication...")
start_time = time.time()
c_cpu = a_cpu @ b_cpu
cpu_time = time.time() - start_time
print(f"CPU Time: {cpu_time:.4f} seconds")

if torch.cuda.is_available():
    # Move the matrices to the GPU
    a_gpu = a_cpu.to(device)
    b_gpu = b_cpu.to(device)

    # Warmup step (the first CUDA call usually takes a bit of extra time to initialize)
    _ = a_gpu @ b_gpu
    torch.cuda.synchronize()

    print("\nTiming GPU Matrix Multiplication...")
    start_time = time.time()
    c_gpu = a_gpu @ b_gpu

    # WE MUST WAIT for the GPU to finish before stopping the timer!
    torch.cuda.synchronize()
    gpu_time = time.time() - start_time

    print(f"GPU Time: {gpu_time:.4f} seconds")
    print(f"Speedup: {cpu_time / gpu_time:.1f}x faster on the GPU!")
else:
    print("\nNo GPU available to run the comparison. Connect a GPU to see the magic!")

### Exercise 8: Moving Data
**Task**: Create a large random tensor `x` of shape `(1000, 1000)`. Move it to the `device` object we defined above.
Then, square the entire tensor (`x * x`) and verify the resulting tensor successfully stayed on the correct device.

In [ ]:
# TODO: Create tensor, move to device, and multiply
x = ...
x = ...
y = ...

print("y is on device:", y.device)

### The "Expected Device" Error
A very common error in Deep Learning is trying to perform an operation between two tensors that live on different devices (e.g., trying to add a CPU tensor to a GPU tensor).

PyTorch strict forces you to explicitly move data to the same device before operating on them, primarily because the CPU and GPU access completely separate physical memory banks. Passing data between them over the PCIe bus is slow, so PyTorch refuses to do it implicitly to prevent "silent" performance bottlenecks.

In [ ]:
# EXAMPLE: Device Errors (Uncomment to see the error!)

tensor_cpu = torch.ones(3, 3)
tensor_gpu = torch.ones(3, 3).to(device)

if device.type == 'cuda':
    try:
        # This will CRASH because they are on different devices!
        result = tensor_cpu + tensor_gpu
    except Exception as e:
        print(f"Error caught: {e}")

        # The fix: Move the CPU tensor to the GPU first!
        result = tensor_cpu.to(device) + tensor_gpu
        print("Fixed sum shape:", result.shape)

---

# Part III: Autograd (Automatic Differentiation)

PyTorch's `autograd` package provides automatic differentiation for all operations on Tensors. It is a **define-by-run framework**, which means that your backpropagation computationally is defined by how your code is run dynamically, and that every single iteration can be different!

We've seen basic tensor attributes like `.shape` and `.device`. Two even more crucial attributes for Deep Learning are `.requires_grad` and `.grad`.

`torch.Tensor` is the central class of the package. If you set its attribute `.requires_grad` as `True`, it starts to track all operations on it. When you finish your computation you can call `.backward()` and have all the gradients computed automatically. The gradient for this tensor will be accumulated into its `.grad` attribute.

## 9. Tracking Operations (`.requires_grad` and `.grad_fn`)
Each tensor has a `.grad_fn` attribute that references a `Function` that created the `Tensor` (except for Tensors created manually by the user - their `grad_fn` is `None`). Tensor and Function are interconnected and build up an acyclic graph, that encodes a complete history of computation.

### Some Background

We had seen three important attributes of a tensor before: (recall them)

The two more important attributes are the `.requires_grad` and `.grad` attribute.

`torch.Tensor` is the central class of the package. If you set its attribute `.requires_grad` as `True`, it starts to track all operations on it. When you finish your computation you can call `.backward()` and have all the gradients computed automatically. The gradient for this tensor will be accumulated into `.grad` attribute.

To stop a tensor from tracking history, you can call `.detach()` to detach it from the computation history, and to prevent future computation from being tracked.

To prevent tracking history (and using memory), you can also wrap the code block in `with torch.no_grad():`. This can be particularly helpful when evaluating a model because the model may have trainable parameters with `requires_grad=True`, but for which we don’t need the gradients.

There’s one more class which is very important for autograd implementation - a `Function`.

`Tensor` and `Function` are interconnected and build up an acyclic graph, that encodes a complete history of computation. Each tensor has a `.grad_fn` attribute that references a `Function` that has created the `Tensor` (except for Tensors created by the user - their `grad_fn is None`).

If you want to compute the derivatives, you can call `.backward()` on a `Tensor`. If `Tensor` is a scalar (i.e. it holds one element data), you don’t need to specify any arguments to `backward()`, however if it has more elements, you need to specify a `gradient` argument that is a tensor of matching shape.

Computational Graph
===================

Conceptually, autograd keeps a record of data (tensors) & all executed
operations (along with the resulting new tensors) in a directed acyclic
graph (DAG) consisting of
[Function](https://pytorch.org/docs/stable/autograd.html#torch.autograd.Function)
objects. In this DAG, leaves are the input tensors, roots are the output
tensors. By tracing this graph from roots to leaves, you can
automatically compute the gradients using the chain rule.

In a forward pass, autograd does two things simultaneously:

-   run the requested operation to compute a resulting tensor, and
-   maintain the operation's *gradient function* in the DAG.

The backward pass kicks off when `.backward()` is called on the DAG
root. `autograd` then:

-   computes the gradients from each `.grad_fn`,
-   accumulates them in the respective tensor's `.grad` attribute, and
-   using the chain rule, propagates all the way to the leaf tensors.

Below is a visual representation of the DAG in our example. In the
graph, the arrows are in the direction of the forward pass. The nodes
represent the backward functions of each operation in the forward pass.
The leaf nodes in blue represent our leaf tensors `a` and `b`.

![](https://pytorch.org/tutorials/_static/img/dag_autograd.png)

<div style="background-color: #54c7ec; color: #fff; font-weight: 700; padding-left: 10px; padding-top: 5px; padding-bottom: 5px"><strong>NOTE:</strong></div>

<div style="background-color: #f3f4f7; padding-left: 10px; padding-top: 10px; padding-bottom: 10px; padding-right: 10px">

<p>An important thing to note is that the graph is recreated from scratch; after each<code>.backward()</code> call, autograd starts populating a new graph. This isexactly what allows you to use control flow statements in your model;you can change the shape, size and operations at every iteration ifneeded.</p>

</div>

Exclusion from the DAG
----------------------

`torch.autograd` tracks operations on all tensors which have their
`requires_grad` flag set to `True`. For tensors that don't require
gradients, setting this attribute to `False` excludes it from the
gradient computation DAG.

The output tensor of an operation will require gradients even if only a
single input tensor has `requires_grad=True`.


In [ ]:
# EXAMPLE: Tracking history of a tensor
x = torch.ones(2, 2, requires_grad=True)

print("x:\n", x)
print("x.requires_grad:", x.requires_grad)
print("x.grad_fn:", x.grad_fn) # Created by user, so it's None

# Let's do a tensor operation
y = x + 2
print("\ny:\n", y)
print("y.grad_fn:", y.grad_fn) # y was created as a result of an operation, so it has a grad_fn!

# Doing more operations
z = y * y * 3
out = z.mean()

print("\nz:\n", z)
print("z.grad_fn:", z.grad_fn)
print("\nout:\n", out)
print("out.grad_fn:", out.grad_fn)

### Leaf vs. Non-Leaf (Intermediate) Tensors
If you look closely at the output above, `x` doesn't have a `.grad_fn`, but `y`, `z`, and `out` do. This introduces a major concept in PyTorch's computational graph:
- **Leaf Tensors**: Tensors created directly by the user (like `x`, or the weights of a neural network). They are the "roots" of the tree. `is_leaf` is `True`. PyTorch **keeps** the calculated gradients (`.grad`) for leaf tensors after `.backward()` is called.
- **Non-Leaf (Intermediate) Tensors**: Tensors created as a result of mathematical operations (like `y` or `z`). PyTorch tracks them to calculate gradients using the chain rule, but by default, it **deletes** their `.grad` to save memory once `.backward()` finishes!

In [ ]:
# EXAMPLE: Leaf vs Non-Leaf Behavior
print("Is x a leaf tensor?", x.is_leaf) # True (created by us)
print("Is y a leaf tensor?", y.is_leaf) # False (result of x + 2)

# Let's see what happens to their gradients during backprop!
# We'll create a fresh graph:
a = torch.tensor([2.0], requires_grad=True) # LEAF
b = a ** 2                                  # NON-LEAF

# If we really want to keep the gradient of an intermediate tensor, we use .retain_grad()
b.retain_grad()

c = b * 3                                   # NON-LEAF

# Trigger backprop
c.backward()

print("\nGradient of 'a' (Leaf):", a.grad)      # PyTorch keeps this!
print("Gradient of 'b' (Non-Leaf):", b.grad)  # Deleted normally, but we used retain_grad()!
print("Gradient of 'c' (Non-Leaf):", c.grad)  # Warning emitted natively: accesses non-leaf grad -> None!

### Visualizing the Computation Graph
Understanding this dynamically built graph is so important that researchers often visualize it. We can use `torchviz` to plot the exact chain of operations leading to `out`.

**If you are in Colab**, `torchviz` and `graphviz` usually work out of the box after a quick pip install!

In [ ]:
# You may need to install torchviz:
!pip install torchviz
import contextlib

try:
    from torchviz import make_dot
    # Visualizing the computation graph for `out`
    display(make_dot(out, params={'x': x}))
except ImportError:
    print("Please install torchviz using `!pip install torchviz` to view the computational graph.")

### A More Complex Computation Graph
The previous graph was just a straight line. Real neural networks have branching paths and multiple inputs converging.
Let's build a tiny graph simulating a single neuron calculating Mean Squared Error: `loss = (w * x + b - y)**2`.

This graph will feature:
* **Multiple Leaf Nodes tracking gradients:** Our parameters `w_weight` and `b_bias`.
* **Leaf Nodes NOT tracking gradients:** Our raw data `x_in` and `y_target`.
* **Branching/Converging:** The addition and multiplication operations will combine these nodes, creating a non-linear tree-like structure.

In [ ]:
# Use completely new variable names so we don't mess up our previous tracking!
w_weight = torch.tensor([2.0], requires_grad=True)
b_bias = torch.tensor([-1.0], requires_grad=True)

# Input data usually doesn't need gradients!
x_in = torch.tensor([3.0])
y_target = torch.tensor([4.0])

# --- Forward Pass (Building the Graph) ---
# 1. Multiplication
weighted_input = w_weight * x_in
# 2. Addition
y_pred = weighted_input + b_bias
# 3. Subtraction & Power (Loss)
mse_loss = (y_pred - y_target) ** 2

try:
    from torchviz import make_dot
    # Visualize the complex converging graph
    dict_params = {'w_weight': w_weight, 'b_bias': b_bias, 'x_in': x_in, 'y_target': y_target}
    display(make_dot(mse_loss, params=dict_params))
except ImportError:
    print("Complex graph created successfully! Install torchviz to view it.")

## 10. Gradients and Backpropagation `.backward()`

Let’s backprop now. Because `out` contains a single scalar, `out.backward()` is equivalent to `out.backward(torch.tensor(1.))`. This calculates the gradients $\frac{d(out)}{dx}$ and stores it in `x.grad`.

In [ ]:
print("Gradient of x before backward:", x.grad) # None

# Backpropagate!
out.backward()

# Print gradients d(out)/dx
print("Gradient of x after backward:\n", x.grad)

# Let's inspect the grad_fn after calling backward()
print("\n--- After backward() ---")
print("x.grad_fn:", x.grad_fn) # Still None, x is a leaf
print("out.grad_fn:", out.grad_fn) # The function reference visually still exists...

# BUT, the graph buffers connecting them are destroyed!
try:
    print("\nAttempting to call backward() a second time...")
    out.backward()
except RuntimeError as e:
    print("Caught RuntimeError:", e)

# Important Note: PyTorch accumulates gradients by default on leaf tensors.
# To prevent this (e.g. at the start of a training loop iteration), we zero them out:
x.grad.data.zero_()
print("\nGradient of x after zero_():\n", x.grad)

### Graph Destruction
Did you see the error above? By default, when `.backward()` completes, PyTorch **destroys the computation graph** (frees the saved intermediate buffers and tensors) to save memory.

The `.grad_fn` property on `out` still points to the operation name (`MeanBackward0`), but the actual data linking `out` back to `x` is gone. If you ever need to call `.backward()` multiple times on the same graph (rare, but sometimes used in GANs or advanced architectures), you must explicitly tell PyTorch to keep it alive using `out.backward(retain_graph=True)`.

### Disabling Autograd
When evaluating or deploying a neural network (making predictions), you do **not** need to calculate gradients. Tracked tensors build a computational graph in the background, consuming extra memory and time.

You should temporarily disable autograd during evaluation/testing using the `with torch.no_grad():` context manager, or detach an individual tensor permanently from the graph using `.detach()`.

In [ ]:
# EXAMPLE: Disabling Autograd
print("Original tensor requires_grad:", x.requires_grad)

# Using set_requires_grad_()
x.requires_grad_(True) # Just ensuring it's True
print("After requires_grad_():", x.requires_grad)

# Using torch.no_grad()
with torch.no_grad():
    # PyTorch ignores operations inside this block
    z2 = x ** 2
    print("\nInside torch.no_grad(), requires_grad:", z2.requires_grad)

# Using .detach()
x_detached = x.detach()
print("\nDetached tensor requires_grad:", x_detached.requires_grad)
print("Are values equal?", x.eq(x_detached).all().item())

### Exercise 9: Computing Gradients of a Matrix
**Task**: Let's simulate a tiny neural network layer! You have a weight matrix `W` and an input vector `x`.
1. Calculate the matrix multiplication (dot product) $z = W @ x$.
2. Calculate the sum of all elements in `z` and call it `loss`.
3. Use `autograd` to find the derivative of the `loss` with respect to `W`.

*Hint*: Remember to set `requires_grad=True` **ONLY** on the weights `W`. In real scenarios we usually only want to calculate gradients for the weights to update them, not the raw input data!

In [ ]:
# DO NOT CHANGE input x
x = torch.tensor([1.0, 2.0, 3.0])

# TODO: Initialize W as a 3x3 float32 tensor of ones that tracks gradients.
W = ...

# TODO: Compute z = W @ x
z = ...

# TODO: Compute the sum of z
loss = ...

# TODO: Trigger backpropagation to compute gradients
...

print("Gradients of W:\n", W.grad)

### Exercise 10: Gradient Accumulation and Retaining the Graph
**Task**: Calculate `loss1 = w ** 2` and `loss2 = w ** 3` where `w = 2.0`.
Call `.backward()` on `loss1`, then call `.backward()` on `loss2`.

Notice that PyTorch throws a graph destruction error unless you use `retain_graph=True` on the first backward! Also, print the gradient of `w` after the second backward. Did the gradients overwrite each other, or accumulate?

In [ ]:
w = torch.tensor([2.0], requires_grad=True)

# TODO: Calculate loss1 and loss2
loss1 = ...
loss2 = ...

# TODO: Call backward on loss1 (What argument do you need to stop graph destruction?)
...
print("Grad after loss1:", w.grad)

# TODO: Call backward on loss2
...
print("Grad after loss2:", w.grad) # Should be 2*w + 3*w**2 = 4 + 12 = 16!

### Exercise 11: A Manual Gradient Descent Update
**Task**: The entire point of autograd is to update our weights to minimize loss.
Given a weight `W`, perform one step of Gradient Descent: `W_new = W - (learning_rate * W.grad)`.

*Crucial Rule*: When updating weights intuitively, PyTorch will crash if you try to mutate a tensor that is currently tracking history (because the update itself becomes part of the computation graph!). You *must* isolate the update inside a `with torch.no_grad():` block! Then, remember to zero the gradient for the next theoretical step!

In [ ]:
W = torch.tensor([5.0], requires_grad=True)
learning_rate = 0.1

loss = (W - 3) ** 2 # Minimum of this function should be at W=3
loss.backward()
print("Gradient:", W.grad)

# TODO: Update W using the gradient formula inside a no_grad() block
...

# TODO: Zero the gradient of W using the .zero_() method on the grad data
...

print("Updated W:", W)
print("Cleared Gradient:", W.grad)